# pdll quickstart: Pairwise Difference Learning in five minutes

**Pairwise Difference Learning (PDL)** trains a model on *pairs* of rows: "do these two rows have the same label?" (or, for regression, "how far apart are their targets?"). A new row is then compared with the training rows, whose labels are known, and the answers are combined. With n training rows the model can learn from up to n² pairs, which is why it is aimed at small tables. [`pdll`](https://github.com/Karim-53/pdll) wraps any scikit-learn model with the usual `fit` / `predict` API. Method: [Belaid, Rabus & Hüllermeier, *Pairwise Difference Learning for Classification*, Discovery Science 2024](https://doi.org/10.1007/978-3-031-78980-9_18). *Disclosure: I am one of the authors.*

In [ ]:
!pip install -q -U pdll

**Wrap any scikit-learn model.** A single decision tree on 300 handwritten digits, alone and wrapped in PDL (same shuffled 5-fold cross-validation for both):

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier

from pdll import PairwiseDifferenceClassifier

X, y = load_digits(return_X_y=True)
subset = np.random.RandomState(0).permutation(len(y))[:300]      # a small dataset
X, y = X[subset], y[subset]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
tree = DecisionTreeClassifier(random_state=0)
print('decision tree      :', cross_val_score(tree, X, y, cv=cv).mean().round(3))
print('PDL(decision tree) :', cross_val_score(PairwiseDifferenceClassifier(tree), X, y, cv=cv).mean().round(3))

**Raw DataFrames** work as they are: strings, categories and missing values (Titanic from OpenML).

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.ensemble import HistGradientBoostingClassifier

X, y = fetch_openml('titanic', version=1, as_frame=True, return_X_y=True)
X = X[['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']]    # drop identifiers such as name or ticket

hgb = HistGradientBoostingClassifier(class_weight='balanced', categorical_features='from_dtype', random_state=0)
print('HistGradientBoosting      :', cross_val_score(hgb, X, y, cv=cv).mean().round(3))
print('PDL(HistGradientBoosting) :', cross_val_score(PairwiseDifferenceClassifier(hgb, random_state=0), X, y, cv=cv).mean().round(3))

PDL does not win everywhere: it helps tree models most, on small to medium tabular data. See the [benchmark on 130 real datasets](https://github.com/Karim-53/pdll/tree/main/benchmark/real_world).

If PDL is useful in your work, please cite the paper:

```bibtex
@inproceedings{belaid2024pairwise,
  title     = {Pairwise Difference Learning for Classification},
  author    = {Belaid, Mohamed Karim and Rabus, Maximilian and H{\"u}llermeier, Eyke},
  booktitle = {Discovery Science},
  pages     = {284--299},
  publisher = {Springer},
  year      = {2024},
  doi       = {10.1007/978-3-031-78980-9_18}
}
```